# Critical Input DEQN: Commitment

This notebook trains the commitment optimal-policy DEQN network. The state is augmented with promise variables attached to forward-looking implementability constraints.

In [ ]:
# Configure paths and commitment-policy training settings.
from pathlib import Path
import json
import subprocess
import sys
import torch

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent

ARTIFACT_ROOT = ROOT / 'baseline_artifacts' / 'critical_input_deqn'
OUT = ARTIFACT_ROOT / 'commitment'
OUT.mkdir(parents=True, exist_ok=True)

STEPS = 8_000
QMC_TRAIN = 256
QMC_VAL = 512
N_VAL_STATES = 1024
HIDDEN_WIDTH = 192
HIDDEN_DEPTH = 2
PROMISE_INIT_SCALE = 1.0
LOG_EVERY = 100
BATCH_SIZE = 2048
SIM_BATCH_SIZE = 512
EPISODE_LENGTH = 20
EPISODE_UPDATES_PER_EPISODE = 2
EPISODE_BROAD_SHARE = 0.50
CHECKPOINT_EVERY = 1000
TARGET_RMS = None
TARGET_MAX_ABS = None
EARLY_STOP_PATIENCE = None
MIN_STEPS_BEFORE_STOP = None
STOP_VAL_STATES = 512
SCENARIO_Q_WEIGHT = 25.0
CALM_ANCHOR_WEIGHT = 5.0
SCENARIO_BURNIN = 5
SCENARIO_HORIZON = 10
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = 'float64'
print(ROOT)
print(OUT)

# Stream subprocess output line by line in Colab instead of waiting silently.
def run_stream(cmd, *, cwd=ROOT, env=None):
    print('Running:', ' '.join(map(str, cmd)), flush=True)
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    ret = proc.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, cmd)



In [ ]:
# Train the commitment network with nonzero promise initialization.
cmd = [
    sys.executable, '-u', '-m', 'src.critical_input_deqn.run_optimal',
    '--output-dir', str(OUT),
    '--kind', 'commitment',
    '--steps', str(STEPS),
    '--qmc-train', str(QMC_TRAIN),
    '--qmc-val', str(QMC_VAL),
    '--n-val-states', str(N_VAL_STATES),
    '--hidden-width', str(HIDDEN_WIDTH),
    '--hidden-depth', str(HIDDEN_DEPTH),
    '--device', DEVICE,
    '--dtype', DTYPE,
    '--promise-init-scale', str(PROMISE_INIT_SCALE),
    '--stop-val-states', str(STOP_VAL_STATES),
    '--log-every', str(LOG_EVERY),
    '--batch-size', str(BATCH_SIZE),
    '--sim-batch-size', str(SIM_BATCH_SIZE),
    '--episode-length', str(EPISODE_LENGTH),
    '--episode-updates-per-episode', str(EPISODE_UPDATES_PER_EPISODE),
    '--episode-broad-share', str(EPISODE_BROAD_SHARE),
    '--checkpoint-every', str(CHECKPOINT_EVERY),
    '--scenario-q-weight', str(SCENARIO_Q_WEIGHT),
    '--calm-anchor-weight', str(CALM_ANCHOR_WEIGHT),
    '--scenario-burnin', str(SCENARIO_BURNIN),
    '--scenario-horizon', str(SCENARIO_HORIZON),
]
if TARGET_RMS is not None:
    cmd += ['--target-rms', str(TARGET_RMS)]
if TARGET_MAX_ABS is not None:
    cmd += ['--target-max-abs', str(TARGET_MAX_ABS)]
if EARLY_STOP_PATIENCE is not None:
    cmd += ['--early-stop-patience', str(EARLY_STOP_PATIENCE)]
if MIN_STEPS_BEFORE_STOP is not None:
    cmd += ['--min-steps-before-stop', str(MIN_STEPS_BEFORE_STOP)]
run_stream(cmd, cwd=ROOT)

In [ ]:
# Inspect out-of-sample residual diagnostics for commitment.
with (OUT / 'commitment_eval.json').open('r', encoding='utf-8') as fh:
    commitment_eval = json.load(fh)
commitment_eval